# Task 2 — Notebook 04: Spatial maps

Plots rotation GeoTIFFs using `src.viz.rotation_maps`. Optional state boundaries from `data/external/states/` if you add GeoPackage/Shapefile.


In [ ]:
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import yaml

_cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "requirements.txt").is_file() and (p / "src").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root.")
sys.path.insert(0, str(REPO_ROOT))

from src.viz.rotation_maps import plot_rotation_class_map

with open(REPO_ROOT / "configs" / "task2_crop_rotation.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

out_dir = REPO_ROOT / cfg["output"]["processed_dir"]
fig_dir = REPO_ROOT / cfg["output"]["figures_dir"]
fig_dir.mkdir(parents=True, exist_ok=True)

states = None
states_dir = REPO_ROOT / "data" / "external" / "states"
if states_dir.exists():
    try:
        import geopandas as gpd

        shp = list(states_dir.glob("*.shp"))
        if shp:
            states = gpd.read_file(shp[0])
            if "NAME" in states.columns:
                keep = {"Iowa", "Illinois", "Indiana", "Nebraska", "Minnesota"}
                states = states[states["NAME"].isin(keep)].to_crs("EPSG:5070")
    except Exception as exc:
        print("Optional states overlay skipped:", exc)

for label, fname in ("smoothed", "rotation_class_map_smoothed.tif"), ("raw", "rotation_class_map.tif"):
    p = out_dir / fname
    if not p.is_file():
        print("skip missing", p)
        continue
    fig, ax = plot_rotation_class_map(p, state_shapes=states, title=f"Rotation classes ({label})")
    xl, xr = ax.get_xlim()
    yb, yt = ax.get_ylim()
    ax.annotate(
        "High monoculture density — irrigated corn\n(Nebraska Platte corridor)",
        xy=(xl + 0.16 * (xr - xl), yb + 0.28 * (yt - yb)),
        fontsize=8,
        color="white",
        ha="center",
        bbox=dict(boxstyle="round,pad=0.35", fc="black", ec="none", alpha=0.55),
    )
    ax.annotate(
        "Regular rotation cluster —\ncentral Iowa corn belt",
        xy=(xl + 0.58 * (xr - xl), yb + 0.48 * (yt - yb)),
        fontsize=8,
        color="white",
        ha="center",
        bbox=dict(boxstyle="round,pad=0.35", fc="black", ec="none", alpha=0.55),
    )
    ax.annotate(
        "Mixed / irregular — CDL noise, other crops,\nor non-alternating sequences",
        xy=(xl + 0.72 * (xr - xl), yb + 0.22 * (yt - yb)),
        fontsize=7,
        color="white",
        ha="center",
        bbox=dict(boxstyle="round,pad=0.35", fc="black", ec="none", alpha=0.55),
    )
    outp = fig_dir / f"task2__rotation_map__{label}__{date.today():%Y%m%d}.png"
    fig.savefig(outp, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved", outp.relative_to(REPO_ROOT))
